# Day 8f — Rule-based flags (alongside the ML score, not blended into it)

Real payment-fraud systems layer simple, deterministic, explainable rules **next to** the ML score — not fused into one number (see `docs/hybrid-merge.md` for the research). Three rules, computed causally (each transaction only compared against what happened *before* it — the same standard a live production system would actually have to meet):

1. **High card velocity** — 3+ transactions from the same card in the preceding hour
2. **New card, high value** — the card's first-ever transaction, and the amount is unusually high
3. **Amount spike** — the amount is far above this card's own typical spend so far

**Same discipline as the graph work**: rules are built without ever looking at `isFraud`. The label is used only afterward, to check whether these blind rules actually catch real fraud.

In [1]:
import sys, json
sys.path.append('..')

import numpy as np
import pandas as pd

from src.data import load_merged_train
from src.split import apply_locked_split
from src.rules import apply_all_rules

train_full = load_merged_train()
train_df, holdout_df = apply_locked_split(train_full)

# Threshold computed from TRAIN-ONLY data (same discipline as everywhere else
# in this project -- never let the holdout influence a threshold used to score it).
amount_threshold = train_df['TransactionAmt'].quantile(0.90)
print(f'New-card high-value threshold (90th percentile of train TransactionAmt): {amount_threshold:.2f}')

New-card high-value threshold (90th percentile of train TransactionAmt): 275.99


## 1. Apply the rules to the full dataset

In [2]:
rule_flags = apply_all_rules(train_full, amount_threshold_for_new_card=amount_threshold)

for col in ['rule_high_velocity', 'rule_new_card_high_value', 'rule_amount_spike', 'any_rule_triggered']:
    print(f'{col}: {rule_flags[col].mean():.4f} trigger rate ({rule_flags[col].sum():,} transactions)')

rule_high_velocity: 0.1476 trigger rate (87,176 transactions)
rule_new_card_high_value: 0.0025 trigger rate (1,460 transactions)
rule_amount_spike: 0.0519 trigger rate (30,670 transactions)
any_rule_triggered: 0.1938 trigger rate (114,457 transactions)


## 2. Now check against real labels: do these blind rules actually catch fraud?

In [3]:
population_fraud_rate = train_full['isFraud'].mean()
print(f'Population fraud rate: {population_fraud_rate:.4f}\n')

summary_rows = []
for col in ['rule_high_velocity', 'rule_new_card_high_value', 'rule_amount_spike', 'any_rule_triggered']:
    mask = rule_flags[col]
    n = mask.sum()
    fraud_rate = train_full.loc[mask, 'isFraud'].mean() if n > 0 else float('nan')
    capture = train_full.loc[mask, 'isFraud'].sum() / train_full['isFraud'].sum()
    summary_rows.append({
        'rule': col,
        'n_triggered': int(n),
        'fraud_rate': fraud_rate,
        'lift_vs_population': fraud_rate / population_fraud_rate,
        'capture_rate': capture,
    })

summary = pd.DataFrame(summary_rows).set_index('rule')
summary

Population fraud rate: 0.0350



,n_triggered,fraud_rate,lift_vs_population,capture_rate
rule,,,,
rule_high_velocity,87176,0.038382,1.096945,0.161932
rule_new_card_high_value,1460,0.045890,1.311529,0.003243
rule_amount_spike,30670,0.047082,1.345579,0.069883
any_rule_triggered,114457,0.040609,1.160592,0.224943


## 3. Do these rules catch anything the classifier (v3.1) already misses?

In [4]:
import joblib
from src.features import get_feature_lists
from src.graph import load_graph
from src.hybrid import assign_cluster_categories
import networkx as nx

# Rebuild v3.1's features to get its holdout predictions (same recipe as notebook 08e)
identity_graph = load_graph('../data/processed/identity_graph.pkl')
behavioral_graph = load_graph('../data/processed/behavioral_graph.pkl')
combined_graph = load_graph('../data/processed/combined_graph.pkl')
with open('../data/processed/cluster_partition.json') as f:
    partition = {int(k): v for k, v in json.load(f).items()}

degree = dict(combined_graph.degree())
weighted_degree = dict(combined_graph.degree(weight='weight'))
pagerank = nx.pagerank(combined_graph, weight='weight')
ring_categories = assign_cluster_categories(train_full['TransactionID'], partition, identity_graph, behavioral_graph)
cluster_sizes = pd.Series(partition).value_counts()

train_full_v31 = train_full.set_index('TransactionID', drop=False)
train_full_v31['ring_category'] = ring_categories
train_full_v31['ring_cluster_size'] = train_full_v31['TransactionID'].map(partition).map(cluster_sizes)
train_full_v31['graph_degree'] = train_full_v31['TransactionID'].map(degree)
train_full_v31['graph_weighted_degree'] = train_full_v31['TransactionID'].map(weighted_degree)
train_full_v31['graph_pagerank'] = train_full_v31['TransactionID'].map(pagerank)
train_full_v31 = train_full_v31.reset_index(drop=True)

numeric_features, categorical_features = get_feature_lists(train_full_v31)
numeric_features = numeric_features + ['ring_cluster_size', 'graph_degree', 'graph_weighted_degree', 'graph_pagerank']
categorical_features = categorical_features + ['ring_category']

_, holdout_df_v31 = apply_locked_split(train_full_v31)
print('Rebuilt v3.1 features for holdout scoring.')

Rebuilt v3.1 features for holdout scoring.


In [5]:
with open('../results/v3_1_structural_features_metrics.json') as f:
    v31_metrics = json.load(f)
v31_threshold = v31_metrics['threshold']

# v3.1's model wasn't persisted to disk in notebook 08e -- retrain identically (same seed, deterministic)
from src.model import build_xgb_pipeline

train_df_v31, _ = apply_locked_split(train_full_v31)
X_train_v31 = train_df_v31[numeric_features + categorical_features]
y_train_v31 = train_df_v31['isFraud']
scale_pos_weight = (y_train_v31 == 0).sum() / (y_train_v31 == 1).sum()

model_v31 = build_xgb_pipeline(numeric_features, categorical_features, scale_pos_weight=scale_pos_weight, random_state=42)
model_v31.fit(X_train_v31, y_train_v31)

X_holdout_v31 = holdout_df_v31[numeric_features + categorical_features]
y_holdout_v31 = holdout_df_v31['isFraud']
classifier_proba_holdout = model_v31.predict_proba(X_holdout_v31)[:, 1]

classifier_missed = (y_holdout_v31.to_numpy() == 1) & (classifier_proba_holdout < v31_threshold)
print(f'Classifier (v3.1) missed {classifier_missed.sum()} real fraud cases in the holdout.')

Classifier (v3.1) missed 1481 real fraud cases in the holdout.


In [6]:
holdout_rule_flags = rule_flags.loc[holdout_df_v31.index]
holdout_any_rule = holdout_rule_flags['any_rule_triggered'].to_numpy()

caught_by_rules = classifier_missed & holdout_any_rule
print(f'Of those {classifier_missed.sum()} classifier misses, {caught_by_rules.sum()} '
      f'({caught_by_rules.sum() / classifier_missed.sum():.1%}) trigger at least one rule flag.')

print('\nWhich rules fire on those caught-by-rules cases:')
for col in ['rule_high_velocity', 'rule_new_card_high_value', 'rule_amount_spike']:
    count = holdout_rule_flags.loc[classifier_missed & holdout_rule_flags[col], col].sum()
    print(f'  {col}: {count}')

Of those 1481 classifier misses, 324 (21.9%) trigger at least one rule flag.

Which rules fire on those caught-by-rules cases:
  rule_high_velocity: 209
  rule_new_card_high_value: 5
  rule_amount_spike: 128


## Save results

In [7]:
rules_results = {
    'population_fraud_rate': population_fraud_rate,
    'amount_threshold_new_card': float(amount_threshold),
    'rule_summary': summary.reset_index().to_dict(orient='records'),
    'classifier_v31_holdout_misses': int(classifier_missed.sum()),
    'caught_by_any_rule': int(caught_by_rules.sum()),
    'caught_by_any_rule_pct_of_misses': float(caught_by_rules.sum() / classifier_missed.sum()),
}

with open('../results/rule_based_flags_metrics.json', 'w') as f:
    json.dump(rules_results, f, indent=2, default=str)

print('Saved to results/rule_based_flags_metrics.json')

Saved to results/rule_based_flags_metrics.json


## Takeaway

_Fill in after running: which rule has the strongest fraud lift, whether the rules genuinely add coverage beyond the classifier and graph signals, and how these should surface in the case queue._